In [ ]:
# ═══════════════════════════════════════════════════
# CELL 0 — SETUP
# ═══════════════════════════════════════════════════
!pip install -q datasets


In [ ]:
# ═══════════════════════════════════════════════════
# CELL 1 — CONFIG
# ═══════════════════════════════════════════════════

# ── Corpus ────────────────────────────────────────
PILE_DATASET  = "monology/pile-uncopyrighted"
TARGET_TOKENS = 50_000_000          # adjustable 10M–100M
CORPUS_CACHE  = "data/pile_tokens.pkl"

# ── Vocabulary ────────────────────────────────────
VOCAB_SIZE = None                   # keep all above MIN_COUNT
MIN_COUNT  = 10

# ── Embedding ─────────────────────────────────────
EMBED_DIM   = 300
MRL_NESTING = [50, 100, 150, 200, 250, 300]

# ── Training ──────────────────────────────────────
WINDOW_SIZE = 5
NEG_SAMPLES = 10
EPOCHS      = 5                     # per call; rerun to continue
BATCH_SIZE  = 32768
LR          = 0.001
SUBSAMPLE_T = 1e-5

# ── Early Stopping ────────────────────────────────
PATIENCE  = 2
MIN_DELTA = 0.01

# ── GloVe ─────────────────────────────────────────
GLOVE_X_MAX = 100
GLOVE_ALPHA = 0.75

# ── FastText ──────────────────────────────────────
FT_NGRAM_LO = 3
FT_NGRAM_HI = 6
FT_BUCKETS  = 200_000

# ── Paths ─────────────────────────────────────────
SAVE_DIR       = "checkpoints"
DATA_DIR       = "data"
STD_W2V_CKPT   = f"{SAVE_DIR}/standard_w2v.pt"
MRL_W2V_CKPT   = f"{SAVE_DIR}/mrl_w2v.pt"
STD_GLOVE_CKPT = f"{SAVE_DIR}/standard_glove.pt"
MRL_GLOVE_CKPT = f"{SAVE_DIR}/mrl_glove.pt"
STD_FT_CKPT    = f"{SAVE_DIR}/standard_fasttext.pt"
MRL_FT_CKPT    = f"{SAVE_DIR}/mrl_fasttext.pt"

print("Config loaded.")


In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2 — DATA
# Pile streaming, Vocabulary, SkipGram/Cooccurrence/FastText datasets
# ═══════════════════════════════════════════════════

import os, pickle, time, threading
from collections import Counter
import numpy as np
import torch
from torch.utils.data import Dataset
from scipy.sparse import coo_matrix, csr_matrix

PAIRS_CACHE = f"{DATA_DIR}/skipgram_pairs.npy"
COOC_CACHE  = f"{DATA_DIR}/cooc.npz"


# ── Corpus download ──────────────────────────────────────────────────────────
def download_corpus(target_tokens=TARGET_TOKENS):
    os.makedirs(DATA_DIR, exist_ok=True)
    if os.path.exists(CORPUS_CACHE):
        print(f"Loading cached tokens from {CORPUS_CACHE}")
        with open(CORPUS_CACHE, "rb") as f:
            tokens = pickle.load(f)
        print(f"  {len(tokens):,} tokens")
        return tokens

    print(f"Streaming {target_tokens/1e6:.0f}M tokens from The Pile …")
    from datasets import load_dataset
    ds = load_dataset(PILE_DATASET, split="train", streaming=True)

    tokens = []
    for doc in ds:
        words = doc["text"].lower().split()
        tokens.extend(words)
        if len(tokens) % 5_000_000 < len(words):
            print(f"  {len(tokens)/1e6:.1f}M …")
        if len(tokens) >= target_tokens:
            break
    tokens = tokens[:target_tokens]

    with open(CORPUS_CACHE, "wb") as f:
        pickle.dump(tokens, f)
    print(f"  Cached {len(tokens):,} tokens")
    return tokens


# ── Vocabulary ───────────────────────────────────────────────────────────────
class Vocabulary:
    def __init__(self, tokens):
        counts = Counter(tokens)
        vocab  = [w for w, c in counts.most_common(VOCAB_SIZE) if c >= MIN_COUNT]
        self.w2i    = {w: i for i, w in enumerate(vocab)}
        self.i2w    = vocab
        self.size   = len(vocab)
        self.counts = np.array([counts[w] for w in vocab], dtype=np.float32)

        freq = self.counts / self.counts.sum()
        self.keep_prob = np.minimum(
            1.0,
            np.sqrt(SUBSAMPLE_T / np.maximum(freq, 1e-12))
            + SUBSAMPLE_T / np.maximum(freq, 1e-12),
        ).astype(np.float32)

    def encode(self, tokens):
        return np.array([self.w2i[t] for t in tokens if t in self.w2i], dtype=np.int32)

    def neg_sample_probs(self):
        p = self.counts ** 0.75
        return (p / p.sum()).astype(np.float32)


# ── FastText Vocabulary (extends Vocabulary with n-gram indices) ─────────────
class FastTextVocabulary(Vocabulary):
    def __init__(self, tokens):
        super().__init__(tokens)
        # Pre-compute padded n-gram hash matrix for all vocab words
        all_hashes = []
        for word in self.i2w:
            bounded = f"<{word}>"
            hashes = []
            for n in range(FT_NGRAM_LO, FT_NGRAM_HI + 1):
                for i in range(len(bounded) - n + 1):
                    hashes.append(hash(bounded[i:i+n]) % FT_BUCKETS)
            all_hashes.append(hashes)

        self.max_ngrams = max(len(h) for h in all_hashes)
        self.ngram_matrix  = np.zeros((self.size, self.max_ngrams), dtype=np.int32)
        self.ngram_lengths = np.zeros(self.size, dtype=np.int32)
        for i, hashes in enumerate(all_hashes):
            self.ngram_matrix[i, :len(hashes)] = hashes
            self.ngram_lengths[i] = len(hashes)
        print(f"  FastText n-grams: max {self.max_ngrams} per word, {FT_BUCKETS} buckets")


# ── Skip-gram pair dataset ──────────────────────────────────────────────────
def build_skipgram_pairs(vocab, tokens):
    """Vectorised skip-gram pair construction with subsampling. Cached to disk."""
    if os.path.exists(PAIRS_CACHE):
        print(f"Loading cached pairs from {PAIRS_CACHE} …")
        pairs = np.load(PAIRS_CACHE, mmap_mode="r")
        print(f"  {len(pairs):,} pairs  ({pairs.nbytes/1e6:.0f} MB)")
        return pairs

    print("Building skip-gram pairs (vectorised) …")
    ids  = vocab.encode(tokens)
    mask = np.random.rand(len(ids)).astype(np.float32) < vocab.keep_prob[ids]
    ids  = ids[mask]
    print(f"  After subsampling: {len(ids):,} tokens")

    chunks = []
    for offset in range(1, WINDOW_SIZE + 1):
        chunks.append(np.stack([ids[:-offset], ids[offset:]], axis=1))
        chunks.append(np.stack([ids[offset:],  ids[:-offset]], axis=1))
    pairs = np.concatenate(chunks, axis=0).astype(np.int32)

    np.save(PAIRS_CACHE, pairs)
    print(f"  {len(pairs):,} pairs  ({pairs.nbytes/1e6:.0f} MB)")
    return pairs


# ── Co-occurrence matrix for GloVe ──────────────────────────────────────────
def build_cooccurrence(vocab, tokens):
    """Build GloVe co-occurrence matrix with 1/distance weighting. No subsampling."""
    if os.path.exists(COOC_CACHE):
        print(f"Loading cached co-occurrence from {COOC_CACHE} …")
        data = np.load(COOC_CACHE)
        print(f"  {len(data['row']):,} non-zero entries")
        return data["row"], data["col"], data["data"]

    print("Building co-occurrence matrix …")
    ids  = vocab.encode(tokens)
    V    = vocab.size
    t0   = time.time()

    accumulated = csr_matrix((V, V), dtype=np.float64)
    for offset in range(1, WINDOW_SIZE + 1):
        weight = 1.0 / offset
        n = len(ids) - offset
        rows = ids[:-offset]
        cols = ids[offset:]
        vals = np.full(n, weight, dtype=np.float64)
        # Forward
        accumulated += coo_matrix((vals, (rows, cols)), shape=(V, V)).tocsr()
        # Backward
        accumulated += coo_matrix((vals, (cols, rows)), shape=(V, V)).tocsr()
        print(f"  offset {offset}/{WINDOW_SIZE} done ({time.time()-t0:.0f}s)")

    cooc = accumulated.tocoo()
    row  = cooc.row.astype(np.int32)
    col  = cooc.col.astype(np.int32)
    data = cooc.data.astype(np.float32)

    np.savez(COOC_CACHE, row=row, col=col, data=data)
    print(f"  {len(row):,} non-zero entries ({time.time()-t0:.0f}s total)")
    return row, col, data


print("Data module loaded.")


In [ ]:
# ═══════════════════════════════════════════════════
# CELL 3 — MODELS
# 6 model classes: Standard/MRL × Word2Vec/GloVe/FastText
# ═══════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F


def neg_sampling_loss(center, context, negatives):
    """
    L = -log σ(v_c · v_+) - Σ log σ(-v_c · v_k)
    center:    (B, d)    context:   (B, d)    negatives: (B, K, d)
    """
    pos_loss = F.logsigmoid((center * context).sum(dim=1))
    neg_loss = F.logsigmoid(
        -torch.bmm(negatives, center.unsqueeze(2)).squeeze(2)
    ).sum(dim=1)
    return -(pos_loss + neg_loss).mean()


# ═══════════════════════════════════════════════════
# WORD2VEC
# ═══════════════════════════════════════════════════

class _BaseW2V(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.W_in  = nn.Embedding(vocab_size, EMBED_DIM)
        self.W_out = nn.Embedding(vocab_size, EMBED_DIM)
        nn.init.uniform_(self.W_in.weight,  -0.5/EMBED_DIM, 0.5/EMBED_DIM)
        nn.init.zeros_(self.W_out.weight)

    def get_embeddings(self):
        return self.W_in.weight.detach().cpu().float().numpy()

    def get_prefix(self, m):
        return self.W_in.weight[:, :m].detach().cpu().float().numpy()


class StandardWord2Vec(_BaseW2V):
    def forward(self, centers, contexts, negatives):
        return neg_sampling_loss(
            self.W_in(centers), self.W_out(contexts), self.W_out(negatives)
        )


class MRLWord2Vec(_BaseW2V):
    """L_MRL = (1/|M|) Σ_{m∈M} L_NS(f(w)[:m], f(c)[:m])"""
    def __init__(self, vocab_size, nesting=None):
        super().__init__(vocab_size)
        self.nesting = nesting or MRL_NESTING
        assert self.nesting[-1] == EMBED_DIM
        self.lam = 1.0 / len(self.nesting)

    def forward(self, centers, contexts, negatives):
        vc = self.W_in(centers)
        vp = self.W_out(contexts)
        vn = self.W_out(negatives)
        loss = vc.new_zeros(1).squeeze()
        for m in self.nesting:
            loss = loss + self.lam * neg_sampling_loss(vc[:,:m], vp[:,:m], vn[:,:,:m])
        return loss


# ═══════════════════════════════════════════════════
# GLOVE
# ═══════════════════════════════════════════════════

class _BaseGloVe(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.W       = nn.Embedding(vocab_size, EMBED_DIM)
        self.W_tilde = nn.Embedding(vocab_size, EMBED_DIM)
        self.b       = nn.Embedding(vocab_size, 1)
        self.b_tilde = nn.Embedding(vocab_size, 1)
        nn.init.uniform_(self.W.weight,       -0.5/EMBED_DIM, 0.5/EMBED_DIM)
        nn.init.uniform_(self.W_tilde.weight, -0.5/EMBED_DIM, 0.5/EMBED_DIM)
        nn.init.zeros_(self.b.weight)
        nn.init.zeros_(self.b_tilde.weight)

    def get_embeddings(self):
        return (self.W.weight + self.W_tilde.weight).detach().cpu().float().numpy()

    def get_prefix(self, m):
        return (self.W.weight[:, :m] + self.W_tilde.weight[:, :m]).detach().cpu().float().numpy()


class StandardGloVe(_BaseGloVe):
    """J = Σ f(X_ij) * (w_i · w̃_j + b_i + b̃_j - log X_ij)²"""
    def forward(self, i_idx, j_idx, x_ij):
        wi = self.W(i_idx)                # (B, D)
        wj = self.W_tilde(j_idx)          # (B, D)
        bi = self.b(i_idx).squeeze(1)     # (B,)
        bj = self.b_tilde(j_idx).squeeze(1)
        dot  = (wi * wj).sum(dim=1)
        diff = dot + bi + bj - torch.log(x_ij)
        weight = torch.clamp(x_ij / GLOVE_X_MAX, max=1.0) ** GLOVE_ALPHA
        return (weight * diff ** 2).mean()


class MRLGloVe(_BaseGloVe):
    def __init__(self, vocab_size, nesting=None):
        super().__init__(vocab_size)
        self.nesting = nesting or MRL_NESTING
        assert self.nesting[-1] == EMBED_DIM
        self.lam = 1.0 / len(self.nesting)

    def forward(self, i_idx, j_idx, x_ij):
        wi = self.W(i_idx)
        wj = self.W_tilde(j_idx)
        bi = self.b(i_idx).squeeze(1)
        bj = self.b_tilde(j_idx).squeeze(1)
        weight = torch.clamp(x_ij / GLOVE_X_MAX, max=1.0) ** GLOVE_ALPHA
        log_x  = torch.log(x_ij)
        loss = wi.new_zeros(1).squeeze()
        for m in self.nesting:
            dot  = (wi[:,:m] * wj[:,:m]).sum(dim=1)
            diff = dot + bi + bj - log_x
            loss = loss + self.lam * (weight * diff ** 2).mean()
        return loss


# ═══════════════════════════════════════════════════
# FASTTEXT
# ═══════════════════════════════════════════════════

class _BaseFT(nn.Module):
    def __init__(self, vocab_size, ngram_matrix, ngram_lengths):
        super().__init__()
        self.W_in  = nn.Embedding(vocab_size, EMBED_DIM)
        self.W_out = nn.Embedding(vocab_size, EMBED_DIM)
        self.W_ng  = nn.Embedding(FT_BUCKETS, EMBED_DIM)
        nn.init.uniform_(self.W_in.weight,  -0.5/EMBED_DIM, 0.5/EMBED_DIM)
        nn.init.zeros_(self.W_out.weight)
        nn.init.uniform_(self.W_ng.weight, -0.5/EMBED_DIM, 0.5/EMBED_DIM)
        # Buffers move to GPU with model.to(device)
        self.register_buffer("ng_matrix",  torch.tensor(ngram_matrix,  dtype=torch.long))
        self.register_buffer("ng_lengths", torch.tensor(ngram_lengths, dtype=torch.long))
        self._max_ng = ngram_matrix.shape[1]

    def _enrich(self, word_ids):
        """word embedding + mean(n-gram embeddings)"""
        word_emb = self.W_in(word_ids)                             # (B, D)
        ng_ids   = self.ng_matrix[word_ids]                        # (B, max_ng)
        ng_lens  = self.ng_lengths[word_ids].unsqueeze(1).float()  # (B, 1)
        ng_emb   = self.W_ng(ng_ids)                               # (B, max_ng, D)
        mask     = torch.arange(self._max_ng, device=word_ids.device) < ng_lens
        ng_emb   = ng_emb * mask.unsqueeze(2)
        return (word_emb + ng_emb.sum(dim=1)) / (1.0 + ng_lens)

    def get_embeddings(self):
        with torch.no_grad():
            ids = torch.arange(self.W_in.weight.shape[0], device=self.W_in.weight.device)
            # Process in chunks to avoid OOM
            embs = []
            for start in range(0, len(ids), 4096):
                embs.append(self._enrich(ids[start:start+4096]).cpu())
            return torch.cat(embs).float().numpy()

    def get_prefix(self, m):
        full = self.get_embeddings()
        return full[:, :m]


class StandardFastText(_BaseFT):
    def forward(self, centers, contexts, negatives):
        return neg_sampling_loss(
            self._enrich(centers), self.W_out(contexts), self.W_out(negatives)
        )


class MRLFastText(_BaseFT):
    def __init__(self, vocab_size, ngram_matrix, ngram_lengths, nesting=None):
        super().__init__(vocab_size, ngram_matrix, ngram_lengths)
        self.nesting = nesting or MRL_NESTING
        assert self.nesting[-1] == EMBED_DIM
        self.lam = 1.0 / len(self.nesting)

    def forward(self, centers, contexts, negatives):
        vc = self._enrich(centers)
        vp = self.W_out(contexts)
        vn = self.W_out(negatives)
        loss = vc.new_zeros(1).squeeze()
        for m in self.nesting:
            loss = loss + self.lam * neg_sampling_loss(vc[:,:m], vp[:,:m], vn[:,:,:m])
        return loss


print("Models loaded.")


In [ ]:
# ═══════════════════════════════════════════════════
# CELL 4 — TRAINING ENGINE
# Model-pair parallelism: Standard on GPU 0, MRL on GPU 1
# Early stopping, checkpoint resume, on-the-fly negative sampling
# ═══════════════════════════════════════════════════
# v3.2 fixes:
#   ▸ Per-batch random sampling (torch.randint) instead of
#     torch.randperm which needs ~1.5-3GB for 189M indices
#   ▸ Zero extra GPU memory beyond pairs + model + optimizer
#   ▸ Batch size 32768 to fit T4 15GB
# ═══════════════════════════════════════════════════

import os, time, threading, pickle, gc
import torch
import numpy as np
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm


class NegativeSampler:
    """On-the-fly negative sampling entirely on GPU."""
    def __init__(self, vocab, device):
        self.probs  = torch.tensor(vocab.neg_sample_probs()).to(device)
        self.device = device

    @torch.no_grad()
    def sample(self, batch_size):
        return torch.multinomial(
            self.probs, num_samples=batch_size * NEG_SAMPLES, replacement=True
        ).view(batch_size, NEG_SAMPLES)


# ── Checkpoint helpers ───────────────────────────────────────────────────────
def _save_ckpt(model, optimizer, scheduler, path, mtype, vsz, ep, best):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "model_type":      mtype,
        "vocab_size":      vsz,
        "epochs_trained":  ep,
        "best_loss":       best,
    }, path)

def _load_ckpt(model, optimizer, scheduler, path, device):
    if not os.path.exists(path):
        return 0, float("inf")
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    if "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    if "scheduler_state" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    return ckpt.get("epochs_trained", 0), ckpt.get("best_loss", float("inf"))


# ── Skip-gram training loop (Word2Vec / FastText) ────────────────────────────
def _train_skipgram(rank, model, pairs_np, vocab, save_path, model_type,
                    epochs=EPOCHS):
    device = torch.device(f"cuda:{rank}")
    torch.cuda.set_device(device)
    model = model.to(device)

    neg_sampler = NegativeSampler(vocab, device)

    # Load pairs to GPU — keep as the only copy, no duplication
    pairs_gpu = torch.tensor(np.array(pairs_np), dtype=torch.long).to(device)
    N         = len(pairs_gpu)
    bs        = BATCH_SIZE
    steps_ep  = N // bs
    total_steps = epochs * steps_ep

    print(f"  [{model_type}@GPU{rank}] Pairs: {N:,} | "
          f"Batch: {bs} | Steps/ep: {steps_ep} | "
          f"GPU mem: {torch.cuda.memory_allocated(device)/1e9:.1f}GB")

    optimizer = Adam(model.parameters(), lr=LR)
    scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=LR*0.01)

    ep_done, best_loss = _load_ckpt(model, optimizer, scheduler, save_path, device)
    if ep_done > 0:
        print(f"  [{model_type}@GPU{rank}] Resumed from epoch {ep_done}")
    else:
        print(f"  [{model_type}@GPU{rank}] Training from scratch")

    patience_ctr = 0
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss, t0 = 0.0, time.time()

        # Per-batch random sampling — O(batch) GPU memory, not O(N)
        # Equivalent to SGD with replacement; standard for word embeddings
        pbar = tqdm(range(steps_ep),
                    desc=f"[{model_type}] Ep {ep_done+epoch}/{ep_done+epochs}",
                    dynamic_ncols=True, position=rank)
        for i in pbar:
            idx       = torch.randint(0, N, (bs,), device=device)
            batch     = pairs_gpu[idx]
            centers   = batch[:, 0]
            contexts  = batch[:, 1]
            negatives = neg_sampler.sample(len(centers))

            optimizer.zero_grad(set_to_none=True)
            loss = model(centers, contexts, negatives)
            loss.backward()
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            if i % 100 == 0:
                pbar.set_postfix(loss=f"{epoch_loss/(i+1):.4f}",
                                 lr=f"{scheduler.get_last_lr()[0]:.2e}")

        avg = epoch_loss / steps_ep
        elapsed = time.time() - t0
        print(f"  [{model_type}@GPU{rank}] Ep {ep_done+epoch} | "
              f"loss {avg:.4f} | {elapsed:.0f}s | "
              f"{steps_ep*bs/elapsed/1e6:.2f}M pairs/s")

        # Early stopping
        if best_loss - avg > MIN_DELTA:
            best_loss = avg
            patience_ctr = 0
        else:
            patience_ctr += 1

        _save_ckpt(model, optimizer, scheduler, save_path,
                   model_type, vocab.size, ep_done + epoch, best_loss)

        if patience_ctr >= PATIENCE:
            print(f"  [{model_type}@GPU{rank}] Early stop at epoch {ep_done+epoch}")
            break

    return model


# ── GloVe training loop ─────────────────────────────────────────────────────
def _train_glove(rank, model, cooc_row, cooc_col, cooc_data, vocab,
                 save_path, model_type, epochs=EPOCHS):
    device = torch.device(f"cuda:{rank}")
    torch.cuda.set_device(device)
    model = model.to(device)

    row_gpu  = torch.tensor(np.array(cooc_row),  dtype=torch.long).to(device)
    col_gpu  = torch.tensor(np.array(cooc_col),  dtype=torch.long).to(device)
    data_gpu = torch.tensor(np.array(cooc_data), dtype=torch.float32).to(device)
    N        = len(row_gpu)
    bs       = BATCH_SIZE
    steps_ep = N // bs
    total_steps = epochs * steps_ep

    print(f"  [{model_type}@GPU{rank}] Co-oc entries: {N:,} | "
          f"GPU mem: {torch.cuda.memory_allocated(device)/1e9:.1f}GB")

    optimizer = Adam(model.parameters(), lr=LR)
    scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=LR*0.01)

    ep_done, best_loss = _load_ckpt(model, optimizer, scheduler, save_path, device)
    if ep_done > 0:
        print(f"  [{model_type}@GPU{rank}] Resumed from epoch {ep_done}")
    else:
        print(f"  [{model_type}@GPU{rank}] Training from scratch")

    patience_ctr = 0
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss, t0 = 0.0, time.time()

        pbar = tqdm(range(steps_ep),
                    desc=f"[{model_type}] Ep {ep_done+epoch}/{ep_done+epochs}",
                    dynamic_ncols=True, position=rank)
        for i in pbar:
            idx = torch.randint(0, N, (bs,), device=device)
            optimizer.zero_grad(set_to_none=True)
            loss = model(row_gpu[idx], col_gpu[idx], data_gpu[idx])
            loss.backward()
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            if i % 100 == 0:
                pbar.set_postfix(loss=f"{epoch_loss/(i+1):.4f}")

        avg = epoch_loss / steps_ep
        elapsed = time.time() - t0
        print(f"  [{model_type}@GPU{rank}] Ep {ep_done+epoch} | "
              f"loss {avg:.4f} | {elapsed:.0f}s")

        if best_loss - avg > MIN_DELTA:
            best_loss = avg
            patience_ctr = 0
        else:
            patience_ctr += 1

        _save_ckpt(model, optimizer, scheduler, save_path,
                   model_type, vocab.size, ep_done + epoch, best_loss)

        if patience_ctr >= PATIENCE:
            print(f"  [{model_type}@GPU{rank}] Early stop at epoch {ep_done+epoch}")
            break

    return model


# ── Pair launcher: Standard on GPU 0, MRL on GPU 1 ──────────────────────────
def train_pair(std_fn, mrl_fn):
    """
    Run two training functions concurrently — one per GPU.
    GIL is released during CUDA ops so both GPUs run at full speed.
    """
    results = [None, None]
    errors  = [None, None]

    def wrapper(idx, fn):
        try:
            results[idx] = fn()
        except Exception as ex:
            errors[idx] = ex
            import traceback; traceback.print_exc()

    t0 = threading.Thread(target=wrapper, args=(0, std_fn))
    t1 = threading.Thread(target=wrapper, args=(1, mrl_fn))
    t0.start(); t1.start()
    t0.join();  t1.join()

    for i, err in enumerate(errors):
        if err:
            raise RuntimeError(f"GPU {i} training failed: {err}")
    return results


print("Training engine loaded.")


In [ ]:
# ═══════════════════════════════════════════════════
# CELL 5 — PREPARE DATA
# Download corpus, build vocab, build and cache all datasets.
# Run once; subsequent runs load from cache.
# ═══════════════════════════════════════════════════

tokens = download_corpus()
vocab  = Vocabulary(tokens)
print(f"Vocabulary: {vocab.size:,} words")

ft_vocab = FastTextVocabulary(tokens)

os.makedirs(SAVE_DIR, exist_ok=True)
with open(f"{SAVE_DIR}/vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)
with open(f"{SAVE_DIR}/ft_vocab.pkl", "wb") as f:
    pickle.dump(ft_vocab, f)

# Build skip-gram pairs (used by Word2Vec + FastText)
pairs = build_skipgram_pairs(vocab, tokens)

# Build co-occurrence matrix (used by GloVe)
cooc_row, cooc_col, cooc_data = build_cooccurrence(vocab, tokens)

print(f"\n✓ Data ready. Pairs: {len(pairs):,} | Co-oc entries: {len(cooc_row):,}")


In [ ]:
# ═══════════════════════════════════════════════════
# CELL 6 — TRAINING CALLS
# 6 individual train functions + 3 pair functions.
# Each resumes from checkpoint. Rerun for more epochs.
# Self-contained: loads data from cache if globals are missing.
# ═══════════════════════════════════════════════════

def _ensure_data():
    """Load vocab, pairs, cooccurrence from cache if not already in globals."""
    global vocab, ft_vocab, pairs, cooc_row, cooc_col, cooc_data

    if "vocab" not in globals() or vocab is None:
        print("Loading vocab from cache …")
        with open(f"{SAVE_DIR}/vocab.pkl", "rb") as f:
            vocab = pickle.load(f)
        print(f"  Vocabulary: {vocab.size:,} words")

    if "ft_vocab" not in globals() or ft_vocab is None:
        print("Loading FastText vocab from cache …")
        with open(f"{SAVE_DIR}/ft_vocab.pkl", "rb") as f:
            ft_vocab = pickle.load(f)

    if "pairs" not in globals() or pairs is None:
        print("Loading skip-gram pairs from cache …")
        pairs = np.load(PAIRS_CACHE, mmap_mode="r")
        print(f"  Pairs: {len(pairs):,}")

    if "cooc_row" not in globals() or cooc_row is None:
        print("Loading co-occurrence from cache …")
        data = np.load(COOC_CACHE)
        cooc_row  = data["row"]
        cooc_col  = data["col"]
        cooc_data = data["data"]
        print(f"  Co-oc entries: {len(cooc_row):,}")


# ── Word2Vec ─────────────────────────────────────────────────────────────────
def train_standard_w2v(epochs=EPOCHS):
    _ensure_data()
    model = StandardWord2Vec(vocab.size)
    return _train_skipgram(0, model, pairs, vocab, STD_W2V_CKPT,
                           "standard_w2v", epochs)

def train_mrl_w2v(epochs=EPOCHS):
    _ensure_data()
    model = MRLWord2Vec(vocab.size)
    return _train_skipgram(1, model, pairs, vocab, MRL_W2V_CKPT,
                           "mrl_w2v", epochs)

def train_w2v_pair(epochs=EPOCHS):
    """Standard W2V on GPU 0 + MRL W2V on GPU 1 simultaneously."""
    _ensure_data()
    print("="*60)
    print("  Word2Vec pair training (Standard ↔ MRL)")
    print("="*60)
    return train_pair(
        lambda: train_standard_w2v(epochs),
        lambda: train_mrl_w2v(epochs),
    )


# ── GloVe ────────────────────────────────────────────────────────────────────
def train_standard_glove(epochs=EPOCHS):
    _ensure_data()
    model = StandardGloVe(vocab.size)
    return _train_glove(0, model, cooc_row, cooc_col, cooc_data, vocab,
                        STD_GLOVE_CKPT, "standard_glove", epochs)

def train_mrl_glove(epochs=EPOCHS):
    _ensure_data()
    model = MRLGloVe(vocab.size)
    return _train_glove(1, model, cooc_row, cooc_col, cooc_data, vocab,
                        MRL_GLOVE_CKPT, "mrl_glove", epochs)

def train_glove_pair(epochs=EPOCHS):
    """Standard GloVe on GPU 0 + MRL GloVe on GPU 1 simultaneously."""
    _ensure_data()
    print("="*60)
    print("  GloVe pair training (Standard ↔ MRL)")
    print("="*60)
    return train_pair(
        lambda: train_standard_glove(epochs),
        lambda: train_mrl_glove(epochs),
    )


# ── FastText ─────────────────────────────────────────────────────────────────
def train_standard_fasttext(epochs=EPOCHS):
    _ensure_data()
    model = StandardFastText(ft_vocab.size, ft_vocab.ngram_matrix,
                             ft_vocab.ngram_lengths)
    return _train_skipgram(0, model, pairs, ft_vocab, STD_FT_CKPT,
                           "standard_fasttext", epochs)

def train_mrl_fasttext(epochs=EPOCHS):
    _ensure_data()
    model = MRLFastText(ft_vocab.size, ft_vocab.ngram_matrix,
                        ft_vocab.ngram_lengths)
    return _train_skipgram(1, model, pairs, ft_vocab, MRL_FT_CKPT,
                           "mrl_fasttext", epochs)

def train_fasttext_pair(epochs=EPOCHS):
    """Standard FastText on GPU 0 + MRL FastText on GPU 1 simultaneously."""
    _ensure_data()
    print("="*60)
    print("  FastText pair training (Standard ↔ MRL)")
    print("="*60)
    return train_pair(
        lambda: train_standard_fasttext(epochs),
        lambda: train_mrl_fasttext(epochs),
    )


print("Training functions ready.")
print("  Individual: train_standard_w2v(), train_mrl_w2v(), etc.")
print("  Pairs:      train_w2v_pair(), train_glove_pair(), train_fasttext_pair()")


In [ ]:
# ═══════════════════════════════════════════════════
# CELL 7 — EXPORT
# Save all 6 embedding matrices + vocab to zip.
# ═══════════════════════════════════════════════════

import zipfile
from IPython.display import FileLink, display


def load_model(save_path):
    """Load model from checkpoint, auto-detecting type and vocab size."""
    ckpt       = torch.load(save_path, map_location="cpu", weights_only=False)
    vocab_size = ckpt["vocab_size"]
    mtype      = ckpt["model_type"]

    # Reconstruct model by type
    if "fasttext" in mtype:
        with open(f"{SAVE_DIR}/ft_vocab.pkl", "rb") as f:
            ftv = pickle.load(f)
        if "mrl" in mtype:
            model = MRLFastText(vocab_size, ftv.ngram_matrix, ftv.ngram_lengths)
        else:
            model = StandardFastText(vocab_size, ftv.ngram_matrix, ftv.ngram_lengths)
    elif "glove" in mtype:
        if "mrl" in mtype:
            model = MRLGloVe(vocab_size)
        else:
            model = StandardGloVe(vocab_size)
    else:  # w2v
        if "mrl" in mtype:
            model = MRLWord2Vec(vocab_size)
        else:
            model = StandardWord2Vec(vocab_size)

    model.load_state_dict(ckpt["model_state"])
    model.eval()
    ep = ckpt.get("epochs_trained", "?")
    bl = ckpt.get("best_loss", "?")
    print(f"  Loaded {mtype} | vocab={vocab_size} | epochs={ep} | best_loss={bl}")
    return model


# ── Export all available models ──────────────────────────────────────────────
CKPT_MAP = {
    "standard_w2v":      STD_W2V_CKPT,
    "mrl_w2v":           MRL_W2V_CKPT,
    "standard_glove":    STD_GLOVE_CKPT,
    "mrl_glove":         MRL_GLOVE_CKPT,
    "standard_fasttext": STD_FT_CKPT,
    "mrl_fasttext":      MRL_FT_CKPT,
}

os.makedirs("exports", exist_ok=True)
exported = {}

for name, ckpt_path in CKPT_MAP.items():
    if os.path.exists(ckpt_path):
        model = load_model(ckpt_path)
        emb   = model.get_embeddings()
        np.save(f"exports/{name}_embeddings.npy", emb)
        exported[name] = emb
        print(f"    {name}: {emb.shape}  ({emb.nbytes/1e6:.1f} MB)")
    else:
        print(f"    {name}: NOT TRAINED (skipped)")

# Save vocab
with open(f"{SAVE_DIR}/vocab.pkl", "rb") as f:
    v = pickle.load(f)
np.save("exports/vocab_words.npy", np.array(v.i2w))
print(f"\nVocab: {len(v.i2w)} words")

# ── Zip everything ───────────────────────────────────────────────────────────
zip_name = "mrl_bias_v3_embeddings.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir("exports"):
        zf.write(f"exports/{fn}", fn)
    zf.write(f"{SAVE_DIR}/vocab.pkl", "vocab.pkl")
    if os.path.exists(f"{SAVE_DIR}/ft_vocab.pkl"):
        zf.write(f"{SAVE_DIR}/ft_vocab.pkl", "ft_vocab.pkl")
print(f"\nEmbeddings zip: {os.path.getsize(zip_name)/1e6:.1f} MB")

# Checkpoints zip
ckpt_zip = "mrl_bias_v3_checkpoints.zip"
with zipfile.ZipFile(ckpt_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(SAVE_DIR):
        fp = os.path.join(SAVE_DIR, fn)
        zf.write(fp, fn)
        print(f"  Added: {fn} ({os.path.getsize(fp)/1e6:.1f} MB)")
print(f"Checkpoints zip: {os.path.getsize(ckpt_zip)/1e6:.1f} MB")

display(FileLink(zip_name))
display(FileLink(ckpt_zip))
